# 02. Build FAISS Vector Index for Job Dataset
**SmartHire GenAI Capstone Project**

This notebook demonstrates:
1. Reading the Kaggle/Naukri job dataset (`data/jobs/jobs.csv`)
2. Generating dense vector embeddings for each job posting
3. Creating an exact Inner Product (Cosine) FAISS index
4. Persisting the index and metadata to disk (`vectorstore/`)
5. Querying the index with sample candidate resumes

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df = pd.read_csv('../data/jobs/jobs.csv')
print(f"Loaded {len(df)} job postings.")
display(df.head(3))

In [ ]:
# Prepare search strings
job_texts = []
for _, row in df.iterrows():
    text = f"Title: {row['title']}. Company: {row['company']}. Skills: {row['skills']}. Description: {row['description']}"
    job_texts.append(text)

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(job_texts, convert_to_numpy=True, normalize_embeddings=True)
print(f"Created embedding matrix of shape: {embeddings.shape}")

In [ ]:
# Create FAISS IndexFlatIP
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

# Save index & metadata
os.makedirs('../vectorstore', exist_ok=True)
faiss.write_index(index, '../vectorstore/jobs_index.faiss')
with open('../vectorstore/jobs_metadata.pkl', 'wb') as f:
    pickle.dump(df, f)

print("FAISS vector index successfully built and saved!")